# Image Classification with a Fully Connected Network

## Images as Data

## The Data

In [ ]:
import io
import tarfile
import urllib.request
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from PIL import Image
from sklearn.model_selection import train_test_split

DATA_DIR = Path("data")
ARCHIVE = DATA_DIR / "UTKFace.tar.gz"
CACHE = DATA_DIR / "utkface_32.npz"
ARCHIVE_URL = "https://huggingface.co/datasets/py97/UTKFace-Cropped/resolve/main/UTKFace.tar.gz"
IMAGE_SIZE = 32

In [ ]:
def build_cache():
    # Read the faces straight from the archive, convert to grayscale and shrink
    DATA_DIR.mkdir(exist_ok=True)
    if not ARCHIVE.exists():
        urllib.request.urlretrieve(ARCHIVE_URL, ARCHIVE)

    images, ages, genders = [], [], []
    with tarfile.open(ARCHIVE) as tar:
        for member in tar:
            if not (member.isfile() and member.name.endswith(".jpg")):
                continue
            fields = member.name.split("/")[-1].split("_")
            try:
                age, gender = int(fields[0]), int(fields[1])
            except ValueError:
                continue
            if gender not in (0, 1):
                continue
            image = Image.open(io.BytesIO(tar.extractfile(member).read()))
            image = image.convert("L").resize((IMAGE_SIZE, IMAGE_SIZE), Image.BILINEAR)
            images.append(np.asarray(image, dtype=np.uint8))
            ages.append(age)
            genders.append(gender)
    np.savez_compressed(CACHE, images=np.stack(images), age=ages, gender=genders)

if not CACHE.exists():
    build_cache()

data = np.load(CACHE)
keep = data["age"] <= 80
images = data["images"][keep]          # (n, 32, 32), uint8
age = data["age"][keep].astype(np.float32)
female = data["gender"][keep].astype(np.float32)

print("images:", images.shape, " share of women:", female.mean().round(3))

In [ ]:
# Flatten, split (stratified by gender) and standardize with training statistics
X_raw = images.reshape(len(images), -1).astype(np.float32) / 255

idx_train, idx_rest = train_test_split(np.arange(len(age)), test_size=0.3, random_state=0, stratify=female)
idx_val, idx_test = train_test_split(idx_rest, test_size=0.5, random_state=0, stratify=female[idx_rest])

pixel_mean = X_raw[idx_train].mean(axis=0)
pixel_std = X_raw[idx_train].std(axis=0) + 1e-6

def standardize(X):
    return (X - pixel_mean) / pixel_std

X_all = torch.tensor(standardize(X_raw))
y_female = torch.tensor(female)
y_age = torch.tensor(age)

print("train / val / test:", len(idx_train), len(idx_val), len(idx_test))

## Gender Classification

### Baselines

### The network

$$
\hat{y}_i = \sigma\left( \varphi(x_i W_h + b_h) W_o + b_o \right), \qquad
L = -\frac{1}{n}\sum_{i=1}^n \left[ y_i \log \hat{y}_i + (1 - y_i)\log (1-\hat{y}_i) \right]
$$

In [ ]:
def fit(model, loss_fn, target, n_epochs=60, learning_rate=1e-3, weight_decay=0.0, batch_size=128, seed=0):
    # Mini-batch Adam; returns the model with the best validation loss and the learning curves
    torch.manual_seed(seed)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    X_train, y_train = X_all[idx_train], target[idx_train]
    X_val, y_val = X_all[idx_val], target[idx_val]

    history = {"train": [], "val": []}
    best_loss, best_state = float("inf"), None
    for epoch in range(n_epochs):
        model.train()
        order = torch.randperm(len(idx_train))
        for start in range(0, len(order), batch_size):
            batch = order[start:start + batch_size]
            optimizer.zero_grad()
            loss_fn(model(X_train[batch]).squeeze(1), y_train[batch]).backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            history["train"].append(loss_fn(model(X_train).squeeze(1), y_train).item())
            history["val"].append(loss_fn(model(X_val).squeeze(1), y_val).item())
        if history["val"][-1] < best_loss:
            best_loss = history["val"][-1]
            best_state = {name: value.clone() for name, value in model.state_dict().items()}
            history["best_epoch"] = epoch

    model.load_state_dict(best_state)
    return model, history

def predict(model, rows):
    model.eval()
    with torch.no_grad():
        return model(X_all[rows]).squeeze(1)

In [ ]:
bce = nn.BCEWithLogitsLoss()
N_HIDDEN = 64

majority_accuracy = max(female[idx_test].mean(), 1 - female[idx_test].mean())

torch.manual_seed(0)
logreg = nn.Linear(X_all.shape[1], 1)
logreg, hist_logreg = fit(logreg, bce, y_female)

torch.manual_seed(0)
gender_net = nn.Sequential(nn.Linear(X_all.shape[1], N_HIDDEN), nn.ReLU(), nn.Linear(N_HIDDEN, 1))
initial_hidden_weights = gender_net[0].weight.detach().clone()   # keep for the weight analysis
gender_net, hist_gender = fit(gender_net, bce, y_female, weight_decay=1e-3)

def accuracy(model, rows):
    return ((predict(model, rows) > 0).float() == y_female[rows]).float().mean().item()

print(f"Majority class : {majority_accuracy:.3f}")
print(f"Logistic regr. : {accuracy(logreg, idx_test):.3f}")
print(f"ANN (k={N_HIDDEN})    : {accuracy(gender_net, idx_test):.3f}   (test accuracy)")

In [ ]:
logits_test = predict(gender_net, idx_test)
pred_test = (logits_test > 0).float()
truth_test = y_female[idx_test]

confusion = np.array([[((truth_test == t) & (pred_test == p)).sum().item() for p in (0, 1)] for t in (0, 1)])
print("rows: truth (M, F), columns: prediction (M, F)")
print(confusion)

wrong = torch.where(pred_test != truth_test)[0]
# the most confident mistakes
worst = wrong[torch.argsort(-logits_test[wrong].abs())][:8]
fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for ax, j in zip(axes, worst):
    i = idx_test[j.item()]
    ax.imshow(images[i], cmap="gray")
    ax.set_title(f"true {'F' if female[i] else 'M'}\n{torch.sigmoid(logits_test[j]).item():.2f} P(F)", fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.show()

## Age Regression

In [ ]:
mse = nn.MSELoss()

torch.manual_seed(0)
age_net = nn.Sequential(nn.Linear(X_all.shape[1], N_HIDDEN), nn.ReLU(), nn.Linear(N_HIDDEN, 1))
age_net, hist_age = fit(age_net, mse, y_age, weight_decay=1e-3)

age_pred = predict(age_net, idx_test)
mae_net = (age_pred - y_age[idx_test]).abs().mean().item()
mae_mean = (y_age[idx_test] - y_age[idx_train].mean()).abs().mean().item()
print(f"MAE of predicting the mean age: {mae_mean:.1f} years")
print(f"MAE of the ANN                : {mae_net:.1f} years")

fig, ax = plt.subplots(figsize=(4.5, 4.5))
ax.scatter(y_age[idx_test], age_pred, s=4, alpha=0.3)
ax.plot([0, 80], [0, 80], color="black", linewidth=1)
ax.set(xlabel="true age", ylabel="predicted age", title="Test set")
plt.show()

## Analysis of the Learned Weights

### Weights Before and After Training

### The Most Influential Neurons

### Comparison with Simple Summaries of the Data

### How Many Different Templates Does the Network Use?

### The Same Analysis for Age

### The Weights are Tied to Pixel Positions

## Summary

## Exercises